# Data EDA

알약 이미지 객체 탐지 프로젝트의 데이터를 확인하고, 학습에 사용하기 전 기본 파일 구조와 데이터 상태를 탐색합니다.


# 1차 EDA 후 데이터 전처리/Dataset 구성에 관해서

- 1차 EDA에서는 데이터 구조, 클래스 분포, bbox 크기/종횡비/위치 분포, 주요 annotation 이상치를 확인하였다. 이후 단계에서는 모델 학습을 위한 데이터셋 로더 파이프라인 설계를 우선 진행한다.

- bbox 좌표 이상치가 1건 확인되었다. 해당 어노테이션을 그대로 진행하면 에러가 날수도 있어 해당 어노테이션 1건은 데이터셋에서 제외하고 진행한다. 2차에서 추가 결측/이상치 발견하고 다 같이 보정예정.

- 데이터 분할은 train 이미지 기준으로 train/validation을 9:1 비율로 나누어 진행한다. 클래스별 데이터 수가 적은 경우가 많아 split 결과에 따른 편차가 발생할 수 있으므로, random seed를 고정하여 재현 가능한 데이터 분할을 사용한다. 다른 split 기법은 증강이나 외부데이터에서 모수를 채울수도 있어서 1차에서는 고려하지 않음.

- transform은 train/validation/test 전부 resize, nomalize, Albumentations(bbox resize)으로 진행. 2차 때 전체 데이터 검수 이후 augmentation 기법 정하고 train에만 적용. validation과 test 데이터에는 augmentation을 적용하지 않고 resize와 normalize만 수행.


## 1. KaggleHub 데이터 다운로드

Kaggle 계정 정보를 입력해 인증하고, 현재 실행 위치에서 위로 올라가며 프로젝트의 `data/raw` 폴더를 찾습니다. 찾은 `data/raw` 아래에 `ai11-level1-project` 폴더를 만들고 KaggleHub의 `output_dir` 옵션으로 competition 데이터를 다운로드합니다.


In [ ]:
# Install kagglehub only once if it is missing.
# Keep this commented during normal reruns to avoid reinstalling packages.
# !uv add kagglehub

import os
from getpass import getpass
from pathlib import Path

# Find the project data/raw directory from the current notebook location.
current_path = Path.cwd().resolve()

for path_candidate in [current_path, *current_path.parents]:
    raw_root_path = path_candidate / "data" / "raw"

    if raw_root_path.exists():
        break
else:
    raise FileNotFoundError("data/raw folder was not found.")

# Reuse the local dataset if it already exists. Download only when missing.
raw_data_path = raw_root_path / "ai11-level1-project"
dataset_path = raw_data_path / "sprint_ai_project1_data"
required_data_dirs = [
    dataset_path / "train_images",
    dataset_path / "test_images",
    dataset_path / "train_annotations",
]

if all(data_dir.exists() for data_dir in required_data_dirs):
    download_path = raw_data_path
    print("Dataset already exists. Skip download.")
else:
    import kagglehub

    raw_data_path.mkdir(parents=True, exist_ok=True)

    # Read Kaggle credentials for this notebook session only.
    os.environ["KAGGLE_USERNAME"] = input("Kaggle Username: ")
    os.environ["KAGGLE_KEY"] = getpass("Kaggle API Token Key: ")

    download_path = kagglehub.competition_download(
        "ai11-level1-project",
        output_dir=str(raw_data_path),
    )

print("Current working directory:", current_path)
print("Raw root path:", raw_root_path)
print("Path to competition files:", download_path)


## 2. 데이터 기본 정보 확인

- 전체 이미지 수는 1,074장입니다.
- 학습 이미지는 232장, 테스트 이미지는 842장입니다.
- 전체 이미지 크기는 모두 976 x 1280으로 동일합니다.
- train annotation JSON 파일은 763개입니다.
- 클래스는 총 56개입니다.
- 원본 class_id는 0부터 시작하는 연속 번호가 아니라 약품 코드 기반 ID입니다.
- YOLO 학습 데이터로 변환할 때는 원본 class_id를 0~55 범위의 연속 class index로 다시 매핑해야 합니다.


In [ ]:
import json

import pandas as pd
from PIL import Image

dataset_path = raw_data_path / "sprint_ai_project1_data"
train_image_path = dataset_path / "train_images"
test_image_path = dataset_path / "test_images"
annotation_path = dataset_path / "train_annotations"

image_paths = sorted([*train_image_path.glob("*.png"), *test_image_path.glob("*.png")])
annotation_paths = sorted(annotation_path.rglob("*.json"))

image_size_rows = []

for image_path in image_paths:
    with Image.open(image_path) as image:
        image_size_rows.append({"width": image.width, "height": image.height})

image_df = pd.DataFrame(image_size_rows)
image_size_df = image_df.groupby(["width", "height"]).size().reset_index(name="image_count")

category_by_id = {}

for annotation_file_path in annotation_paths:
    with annotation_file_path.open("r", encoding="utf-8") as annotation_file:
        annotation_data = json.load(annotation_file)

    for category in annotation_data.get("categories", []):
        category_by_id[category["id"]] = category["name"]

class_df = pd.DataFrame(
    [
        {"class_id": class_id, "class_name": class_name}
        for class_id, class_name in sorted(category_by_id.items())
    ]
)

print(f"Total images: {len(image_paths):,}")
print(f"Train images: {len(list(train_image_path.glob('*.png'))):,}")
print(f"Test images: {len(list(test_image_path.glob('*.png'))):,}")
print(f"Total annotation JSON files: {len(annotation_paths):,}")
print(f"Total classes: {len(class_df):,}")

display(image_size_df)
display(class_df)


## 3. 클래스별 데이터 분포 확인 결과

- 학습 이미지는 232장 중 56개의 클래스가 각 이미지에 몇번 등장한지 확인합니다.
- 어노테이션이 763개라 232장의 이미지에 총 763번 등장합니다.
- 가장 많은 클래스는 `일양하이트린정 2mg`으로 153번 입니다.
- 가장 적은 클래스는 3번만 등장 했습니다.



In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

train_image_paths = sorted(train_image_path.glob("*.png"))

class_image_rows = []

for annotation_file_path in annotation_paths:
    with annotation_file_path.open("r", encoding="utf-8") as annotation_file:
        annotation_data = json.load(annotation_file)

    image_id_to_file_name = {
        image["id"]: image["file_name"] for image in annotation_data.get("images", [])
    }

    for annotation in annotation_data.get("annotations", []):
        class_image_rows.append(
            {
                "class_id": annotation["category_id"],
                "image_file_name": image_id_to_file_name[annotation["image_id"]],
            }
        )

class_image_df = pd.DataFrame(class_image_rows)

class_distribution_df = (
    class_image_df.drop_duplicates(["class_id", "image_file_name"])
    .groupby("class_id")
    .size()
    .reset_index(name="image_count")
    .merge(class_df, on="class_id", how="left")
)

class_distribution_df["image_ratio"] = (
    class_distribution_df["image_count"] / len(train_image_paths)
)
class_distribution_df = class_distribution_df[
    ["class_id", "class_name", "image_count", "image_ratio"]
]
class_distribution_df = class_distribution_df.sort_values("image_count", ascending=False)


display(class_distribution_df)

plot_df = class_distribution_df.sort_values("image_count")

plt.figure(figsize=(10, 14))
plt.barh(plot_df["class_name"], plot_df["image_count"])
plt.title(f"Class Distribution by Image Count (Total train images: {len(train_image_paths):,})")
plt.xlabel("Image count")
plt.ylabel("Class name")
plt.tight_layout()
plt.show()


## 4. 클래스 분포 해석
 
 - 각 클래스 별로 13.6번(763/232) 균등하게 등장하는 것이 모델 학습하기 이상적인 조건이라 생각.
 - KDE plot, Box plot, Deviation 세 시각화 기법으로 확인 진행.
 - KDE plot : 정규분포를 보는 것이기 때문에 균등한것을 보는것과는 맞지않다고 생각.
 - Deviation : 특정 3 클래스 이외에는 13.6번과 많이 차이나지 않는다고 생각됨.
 - Box plot : 특정 3 클래스 이외에는 13.6번과 많이 차이나지 않는다고 생각됨.
 
 - 특정 3 클래스를 제외하고는 균등하게 등장했느냐는 관점에서는 크게 불균형이라고 볼 순 없다.
 - 단, 특정 3클래스와 형태나 색상이 유사한 소수 클래스는 모델이 많이 본 특정 클래스로 오분류될 가능성이 있습니다.
 - 나머지 클래스 들은 애초에 절대적 등장 수가 적기 학습 샘플이 부족해 클래스 고유 특징을 충분히 학습하지 못할 수 있습니다.
 - 따라서 모델 학습 시 다수 클래스 편향과 소수 클래스 성능 저하를 함께 고려해야 합니다.
 - 1차 학습&평가 후 확인 : 클래스 판단이 틀린 것들중 1. 특정 3 클래스와 혼동하는지 2. 데이터 적은 클래스와 관련있는지 확인 
 - 고려해야할 사항 : 색이 비슷한 것은 음각이 잘보이게 해서 구분하기(증강), 데이터 절대값이 작은 클래스는 절대값을 늘릴 수 있는지


In [ ]:
import numpy as np

image_count_values = class_distribution_df["image_count"]

image_count_mean = image_count_values.mean()
image_count_median = image_count_values.median()
image_count_std = image_count_values.std()
image_count_skew = image_count_values.skew()
ideal_count = image_count_values.sum() / len(class_distribution_df)

distribution_summary_df = pd.DataFrame(
    [
        {"metric": "mean", "value": image_count_mean},
        {"metric": "median", "value": image_count_median},
        {"metric": "std", "value": image_count_std},
        {"metric": "skew", "value": image_count_skew},
        {"metric": "ideal_uniform_count", "value": ideal_count},
    ]
)

display(distribution_summary_df)

kde_x = np.linspace(image_count_values.min(), image_count_values.max(), 500)
bandwidth = 1.06 * image_count_std * (len(image_count_values) ** (-1 / 5))
kde_y = np.zeros_like(kde_x)

for image_count in image_count_values:
    kde_y += np.exp(-0.5 * ((kde_x - image_count) / bandwidth) ** 2)

kde_y = kde_y / (len(image_count_values) * bandwidth * np.sqrt(2 * np.pi))

plt.figure(figsize=(10, 6))
plt.hist(image_count_values, bins=20, density=True, alpha=0.6, label="Class count histogram")
plt.plot(kde_x, kde_y, color="black", linewidth=2, label="KDE")
plt.axvline(image_count_mean, color="red", linestyle="--", label=f"Mean: {image_count_mean:.1f}")
plt.axvline(image_count_median, color="blue", linestyle="--", label=f"Median: {image_count_median:.1f}")
plt.axvline(ideal_count, color="green", linestyle="-", label=f"Ideal uniform: {ideal_count:.1f}")
plt.title("Class Image Count Distribution with KDE")
plt.xlabel("Image count per class")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
deviation_df = class_distribution_df.copy()
deviation_df["deviation_from_ideal"] = deviation_df["image_count"] - ideal_count
deviation_df = deviation_df.sort_values("deviation_from_ideal")

bar_colors = deviation_df["deviation_from_ideal"].apply(
    lambda value: "tab:red" if value > 0 else "tab:blue"
)

plt.figure(figsize=(10, 14))
plt.barh(deviation_df["class_name"], deviation_df["deviation_from_ideal"], color=bar_colors)
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Deviation from Ideal Uniform Count ({ideal_count:.1f})")
plt.xlabel("Image count deviation from ideal")
plt.ylabel("Class name")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(5, 7))
plt.boxplot(image_count_values, vert=True)
plt.axhline(ideal_count, color="green", linestyle="-", label=f"Ideal uniform: {ideal_count:.1f}")
plt.axhline(image_count_median, color="blue", linestyle="--", label=f"Median: {image_count_median:.1f}")
plt.title("Class Image Count Box Plot")
plt.ylabel("Image count per class")
plt.xticks([1], ["Classes"])
plt.legend()
plt.tight_layout()
plt.show()


## 5. 클래스 간 시각적 유사성

- 등장빈도 높은 상위 3종 알약 중 선홍색이며 정원형(일양 하이트린 2mg) 흰색이며 타원형(아토젯정10/40mg)에 대해서는 다른 클래스 들과 유사성을 보임
- 두 타입의 알약형태에 대해서는 학습을 잘할지 못할지는 추후확인 


In [ ]:
class_sample_rows = []
seen_class_ids = set()

for annotation_file_path in annotation_paths:
    with annotation_file_path.open("r", encoding="utf-8") as annotation_file:
        annotation_data = json.load(annotation_file)

    image_id_to_file_name = {
        image["id"]: image["file_name"] for image in annotation_data.get("images", [])
    }

    for annotation in annotation_data.get("annotations", []):
        class_id = annotation["category_id"]

        if class_id in seen_class_ids:
            continue

        image_file_name = image_id_to_file_name[annotation["image_id"]]
        class_name = class_df.loc[class_df["class_id"] == class_id, "class_name"].iloc[0]

        class_sample_rows.append(
            {
                "class_id": class_id,
                "class_name": class_name,
                "image_path": train_image_path / image_file_name,
                "bbox": annotation["bbox"],
            }
        )
        seen_class_ids.add(class_id)

    if len(seen_class_ids) == len(class_df):
        break

class_sample_df = pd.DataFrame(class_sample_rows).sort_values("class_id")

cols = 3
rows = int(np.ceil(len(class_sample_df) / cols))

plt.figure(figsize=(15, rows * 4.0))

for plot_index, (_, row) in enumerate(class_sample_df.iterrows(), start=1):
    image = Image.open(row["image_path"]).convert("RGB")
    x, y, width, height = row["bbox"]
    padding = 20

    left = max(0, int(x - padding))
    top = max(0, int(y - padding))
    right = min(image.width, int(x + width + padding))
    bottom = min(image.height, int(y + height + padding))

    crop_image = image.crop((left, top, right, bottom))

    plt.subplot(rows, cols, plot_index)
    plt.imshow(crop_image)
    plt.title(f"{row['class_id']}\n{row['class_name']}", fontsize=18)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 6. bbox 분석 결과 정리

### 주요 용어

- `bbox_width_ratio`: 이미지 전체 너비 대비 bbox 너비 비율
- `bbox_height_ratio`: 이미지 전체 높이 대비 bbox 높이 비율
- `bbox_area_ratio`: 이미지 전체 면적 대비 bbox 면적 비율
- `aspect_ratio`: bbox의 가로/세로 비율, `bbox_width / bbox_height`
- `center_x_ratio`: 이미지 너비 기준 bbox 중심 x좌표 비율
- `center_y_ratio`: 이미지 높이 기준 bbox 중심 y좌표 비율

### bbox 크기 분포(area_ratio)

- 전체 bbox annotation 수는 763개이다.
- 이미지 전체 면적 대비 bbox의 면적비율 
  median 약 0.043 / mean은 약 0.056 
- 이미지 면적 대비 1% 미만의 매우 작은 bbox는 0개이다.
- 이미지 면적 대비 1% 초과 10% 이하 중간 bbox는 680개(약 89.1%)이다.
- 이미지 면적 대비 10%를 초과하는 큰 bbox는 83개이다.

- 1~10%구간 상세는 
- 2% 초과 ~ 3% 이하: 217개
- 4% 초과 ~ 5% 이하: 141개
- 3% 초과 ~ 4% 이하: 106개
- small 객체(1%이하)는 없지만 작은객체 기준에 치우친 medium 객체들이 모여있는 것을 알 수 있다.

### bbox 종횡비 분포(aspect_ratio)

- aspect_ratio의 median은 1.00, mean은 약 0.99로 1에 매우 가깝다.
- square_like(0.8 <= aspect_ratio <= 1.25) bbox는 497개, 전체의 약 65.1%이다.
- wide(aspect_ratio > 1.25) bbox는 97개, 약 12.7%이다.
- tall(tall: aspect_ratio < 0.8) bbox는 169개, 약 22.1%이다.
- 대부분의 bbox는 가로세로 비율이 크게 치우치지 않은 형태이며, 원형 또는 완만한 타원형 알약이 많은 데이터 특성과 연결된다.

### bbox 위치 분포(center_ratio)

- bbox 중심점은 이미지 전체에 무작위로 퍼져 있기보다 특정 위치 패턴에 몰려 있다.
- 4개 알약 이미지는 사각형 배치, 3개 알약 이미지는 삼각형 또는 역삼각형 배치 경향이 보인다.
- 이는 촬영 과정에서 알약을 일정한 규칙으로 배치한 영향으로 해석할 수 있다.
- 실제 서비스 환경처럼 알약 위치가 자유롭게 달라지는 경우에는 위치 다양성이 부족할 가능성이 있다.

### bbox 좌표 이상치

- bbox 좌표 유효성 검사 결과 valid bbox는 762개, invalid bbox는 1개로 확인되었다.
- 이상치 파일은 `K-003351-016262-018357_0_2_0_2_75_000_200.png`이다.
- 해당 bbox는 `bbox_x=6567`, `bbox_width=311`로 기록되어 있으며, 이미지 너비 976을 크게 벗어난다.
- `bbox_x=6567`은 `bbox_x=656`의 오기입 가능성이 높다.
- `bbox_x=656`으로 가정하면 `x_max=967`이 되어 이미지 너비 976 안에 포함된다.
- 시각화 결과에서도 `x=656` 가정 bbox가 오른쪽 아래의 `GLT400` 알약을 적절히 감싸는 것을 확인할 수 있다.

### bbox 종합 해석

- 데이터셋의 객체들은 대부분 이미지 내에서 지나치게 작지 않은 중간 크기 이상으로 나타난다.
- bbox 형태는 정방형에 가까운 경우가 많다. 
- 학습 관점에서는 작은 객체 탐지 자체보다, 유사한 크기와 형태를 가진 알약 클래스 간 세밀한 구분이 더 중요한 과제일 가능성이 높다. 
- bbox 중심 위치가 특정 촬영 배치 패턴에 집중되어 있어, 통제된 촬영 환경의 영향이 크다고 판단된다. 
- 학습 전처리 단계에서 발견된 이상치는 1차에는 무시한채로 진행하고 2차에는 확인된 bbox 좌표 이상치 1개를 보정하는 방향으로 진행한다.

In [ ]:
# 박스 정보를 저장할 리스트를 만듭니다.
bbox_rows = []

# 각 어노테이션 JSON을 읽어 이미지 정보와 박스 정보를 연결합니다.
for annotation_file_path in annotation_paths:
    with annotation_file_path.open("r", encoding="utf-8") as annotation_file:
        annotation_data = json.load(annotation_file)

    image_id_to_info = {
        image["id"]: image for image in annotation_data.get("images", [])
    }

    for annotation in annotation_data.get("annotations", []):
        image_info = image_id_to_info[annotation["image_id"]]
        x, y, bbox_width, bbox_height = annotation["bbox"]
        image_width = image_info["width"]
        image_height = image_info["height"]

        bbox_rows.append(
            {
                "annotation_id": annotation["id"],
                "image_id": annotation["image_id"],
                "image_file_name": image_info["file_name"],
                "class_id": annotation["category_id"],
                "image_width": image_width,
                "image_height": image_height,
                "bbox_x": x,
                "bbox_y": y,
                "bbox_width": bbox_width,
                "bbox_height": bbox_height,
                "bbox_area": bbox_width * bbox_height,
                "aspect_ratio": bbox_width / bbox_height,
                "center_x": x + bbox_width / 2,
                "center_y": y + bbox_height / 2,
                "bbox_width_ratio": bbox_width / image_width,
                "bbox_height_ratio": bbox_height / image_height,
                "bbox_area_ratio": (bbox_width * bbox_height) / (image_width * image_height),
                "center_x_ratio": (x + bbox_width / 2) / image_width,
                "center_y_ratio": (y + bbox_height / 2) / image_height,
            }
        )

# 박스 데이터를 DataFrame으로 만들고 클래스명을 붙입니다.
bbox_df = pd.DataFrame(bbox_rows).merge(class_df, on="class_id", how="left")
bbox_df = bbox_df[
    [
        "annotation_id",
        "image_id",
        "image_file_name",
        "class_id",
        "class_name",
        "image_width",
        "image_height",
        "bbox_x",
        "bbox_y",
        "bbox_width",
        "bbox_height",
        "bbox_area",
        "aspect_ratio",
        "center_x",
        "center_y",
        "bbox_width_ratio",
        "bbox_height_ratio",
        "bbox_area_ratio",
        "center_x_ratio",
        "center_y_ratio",
    ]
]

# 생성된 박스 데이터의 샘플과 전체 개수를 확인합니다.
display(bbox_df.head())
print(f"Total bbox annotations: {len(bbox_df):,}")


In [ ]:
# 박스 좌표가 이미지 범위 안에 있는지 검사합니다.
bbox_validity_df = bbox_df.copy()
bbox_validity_df["is_x_min_valid"] = bbox_validity_df["bbox_x"] >= 0
bbox_validity_df["is_y_min_valid"] = bbox_validity_df["bbox_y"] >= 0
bbox_validity_df["is_x_max_valid"] = (
    bbox_validity_df["bbox_x"] + bbox_validity_df["bbox_width"] <= bbox_validity_df["image_width"]
)
bbox_validity_df["is_y_max_valid"] = (
    bbox_validity_df["bbox_y"] + bbox_validity_df["bbox_height"] <= bbox_validity_df["image_height"]
)
bbox_validity_df["is_center_x_valid"] = bbox_validity_df["center_x_ratio"].between(0, 1)
bbox_validity_df["is_center_y_valid"] = bbox_validity_df["center_y_ratio"].between(0, 1)

# 조건을 하나라도 위반한 박스를 이상치로 분리합니다.
validity_columns = [
    "is_x_min_valid",
    "is_y_min_valid",
    "is_x_max_valid",
    "is_y_max_valid",
    "is_center_x_valid",
    "is_center_y_valid",
]
bbox_validity_df["is_bbox_valid"] = bbox_validity_df[validity_columns].all(axis=1)
invalid_bbox_df = bbox_validity_df[~bbox_validity_df["is_bbox_valid"]].copy()

# 유효한 박스와 이상치 박스 개수를 확인합니다.
bbox_validity_summary_df = pd.DataFrame(
    [
        {"status": "valid", "bbox_count": bbox_validity_df["is_bbox_valid"].sum()},
        {"status": "invalid", "bbox_count": len(invalid_bbox_df)},
    ]
)
bbox_validity_summary_df["bbox_ratio"] = bbox_validity_summary_df["bbox_count"] / len(bbox_validity_df)

display(bbox_validity_summary_df)

# 이상치로 판단된 박스의 세부 정보를 표로 확인합니다.
display(
    invalid_bbox_df[
        [
            "image_file_name",
            "class_id",
            "class_name",
            "image_width",
            "image_height",
            "bbox_x",
            "bbox_y",
            "bbox_width",
            "bbox_height",
            "center_x_ratio",
            "center_y_ratio",
            *validity_columns,
        ]
    ]
)


In [ ]:
# 요약할 박스 크기와 비율 지표를 선택합니다.
bbox_summary_columns = [
    "bbox_width",
    "bbox_height",
    "bbox_area",
    "aspect_ratio",
    "bbox_width_ratio",
    "bbox_height_ratio",
    "bbox_area_ratio",
]

# 선택한 지표들의 기본 통계량과 분위수를 표로 정리합니다.
bbox_summary_df = (
    bbox_df[bbox_summary_columns]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .T
    .reset_index()
    .rename(columns={"index": "metric", "50%": "median"})
)

bbox_summary_df = bbox_summary_df[
    ["metric", "count", "mean", "std", "min", "25%", "median", "75%", "90%", "95%", "max"]
]

display(bbox_summary_df)

# 작은 박스와 큰 박스를 구분할 면적 비율 기준을 정의합니다.
small_bbox_threshold = 0.01
large_bbox_threshold = 0.10

# 기준보다 작거나 큰 박스의 개수를 확인합니다.
small_bbox_count = (bbox_df["bbox_area_ratio"] < small_bbox_threshold).sum()
large_bbox_count = (bbox_df["bbox_area_ratio"] > large_bbox_threshold).sum()

print(f"Small bbox area ratio < {small_bbox_threshold:.0%}: {small_bbox_count:,} / {len(bbox_df):,}")
print(f"Large bbox area ratio > {large_bbox_threshold:.0%}: {large_bbox_count:,} / {len(bbox_df):,}")


In [ ]:
# 박스 면적 비율 분포를 히스토그램으로 확인합니다.
plt.figure(figsize=(10, 5))
plt.hist(bbox_df["bbox_area_ratio"], bins=30, color="tab:blue", alpha=0.75)
plt.axvline(bbox_df["bbox_area_ratio"].median(), color="red", linestyle="--", label=f"Median: {bbox_df['bbox_area_ratio'].median():.3f}")
plt.axvline(bbox_df["bbox_area_ratio"].mean(), color="green", linestyle="-", label=f"Mean: {bbox_df['bbox_area_ratio'].mean():.3f}")
plt.title("BBox Area Ratio Distribution")
plt.xlabel("BBox area ratio")
plt.ylabel("Annotation count")
plt.legend()
plt.tight_layout()
plt.show()

# 박스 너비와 높이 비율 분포를 같은 축에서 비교합니다.
plt.figure(figsize=(10, 5))
plt.hist(bbox_df["bbox_width_ratio"], bins=30, alpha=0.6, label="Width ratio")
plt.hist(bbox_df["bbox_height_ratio"], bins=30, alpha=0.6, label="Height ratio")
plt.title("BBox Width and Height Ratio Distribution")
plt.xlabel("Ratio to image size")
plt.ylabel("Annotation count")
plt.legend()
plt.tight_layout()
plt.show()

# 박스 면적 비율의 중앙값과 이상치 범위를 박스플롯으로 확인합니다.
plt.figure(figsize=(5, 7))
plt.boxplot(bbox_df["bbox_area_ratio"], vert=True)
plt.axhline(bbox_df["bbox_area_ratio"].median(), color="red", linestyle="--", label=f"Median: {bbox_df['bbox_area_ratio'].median():.3f}")
plt.title("BBox Area Ratio Box Plot")
plt.ylabel("BBox area ratio")
plt.xticks([1], ["BBoxes"])
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 박스 종횡비 분포를 히스토그램으로 확인합니다.
plt.figure(figsize=(10, 5))
plt.hist(bbox_df["aspect_ratio"], bins=30, color="tab:purple", alpha=0.75)
plt.axvline(1.0, color="black", linestyle="--", label="Square ratio: 1.0")
plt.axvline(bbox_df["aspect_ratio"].median(), color="red", linestyle="--", label=f"Median: {bbox_df['aspect_ratio'].median():.2f}")
plt.axvline(bbox_df["aspect_ratio"].mean(), color="green", linestyle="-", label=f"Mean: {bbox_df['aspect_ratio'].mean():.2f}")
plt.title("BBox Aspect Ratio Distribution")
plt.xlabel("Aspect ratio (width / height)")
plt.ylabel("Annotation count")
plt.legend()
plt.tight_layout()
plt.show()

# 정방형에 가까운 박스와 가로나 세로로 긴 박스의 비율을 계산합니다.
square_like_count = bbox_df["aspect_ratio"].between(0.8, 1.25).sum()
wide_bbox_count = (bbox_df["aspect_ratio"] > 1.25).sum()
tall_bbox_count = (bbox_df["aspect_ratio"] < 0.8).sum()

aspect_ratio_summary_df = pd.DataFrame(
    [
        {"shape_group": "square_like", "condition": "0.8 <= aspect_ratio <= 1.25", "bbox_count": square_like_count},
        {"shape_group": "wide", "condition": "aspect_ratio > 1.25", "bbox_count": wide_bbox_count},
        {"shape_group": "tall", "condition": "aspect_ratio < 0.8", "bbox_count": tall_bbox_count},
    ]
)
aspect_ratio_summary_df["bbox_ratio"] = aspect_ratio_summary_df["bbox_count"] / len(bbox_df)

display(aspect_ratio_summary_df)

# 박스 너비 비율과 높이 비율의 관계를 산점도로 확인합니다.
plt.figure(figsize=(7, 7))
plt.scatter(
    bbox_df["bbox_width_ratio"],
    bbox_df["bbox_height_ratio"],
    alpha=0.5,
    s=25,
)
plt.plot([0, 1], [0, 1], color="black", linestyle="--", label="width = height")
plt.title("BBox Width Ratio vs Height Ratio")
plt.xlabel("BBox width ratio")
plt.ylabel("BBox height ratio")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 박스 중심점 위치 분포를 히트맵으로 확인합니다.
plt.figure(figsize=(8, 7))
plt.hist2d(
    bbox_df["center_x_ratio"],
    bbox_df["center_y_ratio"],
    bins=20,
    range=[[0, 1], [0, 1]],
    cmap="Blues",
)
plt.colorbar(label="BBox count")

# 이미지 중앙선을 표시해 박스 중심이 어느 위치에 몰리는지 확인합니다.
plt.axvline(0.5, color="red", linestyle="--", linewidth=1, label="Image center")
plt.axhline(0.5, color="red", linestyle="--", linewidth=1)
plt.gca().invert_yaxis()
plt.title("BBox Center Position Heatmap")
plt.xlabel("Center x ratio")
plt.ylabel("Center y ratio")
plt.legend()
plt.tight_layout()
plt.show()

# 중심 좌표의 기본 통계량을 표로 정리합니다.
bbox_center_summary_df = bbox_df[["center_x_ratio", "center_y_ratio"]].describe().T.reset_index()
bbox_center_summary_df = bbox_center_summary_df.rename(
    columns={"index": "metric", "50%": "median"}
)
bbox_center_summary_df = bbox_center_summary_df[
    ["metric", "count", "mean", "std", "min", "25%", "median", "75%", "max"]
]

display(bbox_center_summary_df)


In [ ]:
# 좌표 이상치가 확인된 이미지와 박스 정보를 선택합니다.
target_image_file_name = "K-003351-016262-018357_0_2_0_2_75_000_200.png"
target_bbox_df = bbox_df[bbox_df["image_file_name"] == target_image_file_name].copy()
target_image_path = train_image_path / target_image_file_name

# 이미지 범위를 벗어난 박스만 선택해 x=656 가정값과 비교합니다.
invalid_target_bbox_df = target_bbox_df[
    (target_bbox_df["bbox_x"] + target_bbox_df["bbox_width"] > target_bbox_df["image_width"])
    | (target_bbox_df["bbox_y"] + target_bbox_df["bbox_height"] > target_bbox_df["image_height"])
    | (~target_bbox_df["center_x_ratio"].between(0, 1))
    | (~target_bbox_df["center_y_ratio"].between(0, 1))
].copy()

if len(invalid_target_bbox_df) != 1:
    raise ValueError(f"Expected 1 invalid bbox, found {len(invalid_target_bbox_df)}")

assumed_bbox_x = 656
original_row = invalid_target_bbox_df.iloc[0]
comparison_rows = [
    {
        "case": "original_x_6567",
        "bbox_x": original_row["bbox_x"],
        "bbox_y": original_row["bbox_y"],
        "bbox_width": original_row["bbox_width"],
        "bbox_height": original_row["bbox_height"],
        "edgecolor": "red",
    },
    {
        "case": "assumed_x_656",
        "bbox_x": assumed_bbox_x,
        "bbox_y": original_row["bbox_y"],
        "bbox_width": original_row["bbox_width"],
        "bbox_height": original_row["bbox_height"],
        "edgecolor": "lime",
    },
]
comparison_bbox_df = pd.DataFrame(comparison_rows)
comparison_bbox_df["x_max"] = comparison_bbox_df["bbox_x"] + comparison_bbox_df["bbox_width"]
comparison_bbox_df["y_max"] = comparison_bbox_df["bbox_y"] + comparison_bbox_df["bbox_height"]

# 보정 가정 좌표를 이미지 위에 표시하고 원본 좌표는 범위 밖 좌표로 표에서 확인합니다.
fig, ax = plt.subplots(figsize=(8, 10))
image = Image.open(target_image_path).convert("RGB")
ax.imshow(image)

for _, row in comparison_bbox_df.iterrows():
    if row["x_max"] <= image.width and row["y_max"] <= image.height:
        rectangle = plt.Rectangle(
            (row["bbox_x"], row["bbox_y"]),
            row["bbox_width"],
            row["bbox_height"],
            fill=False,
            edgecolor=row["edgecolor"],
            linewidth=3,
        )
        ax.add_patch(rectangle)
        ax.text(
            row["bbox_x"],
            row["bbox_y"],
            row["case"],
            color="black",
            fontsize=11,
            bbox={"facecolor": row["edgecolor"], "alpha": 0.75, "pad": 3},
        )

ax.set_title(f"Assumed BBox Correction: {target_image_file_name}")
ax.set_xlim(0, image.width)
ax.set_ylim(image.height, 0)
ax.axis("off")
plt.tight_layout()
plt.show()

# 원본값과 보정 가정값이 이미지 범위에 들어오는지 표로 확인합니다.
comparison_bbox_df["is_x_in_image"] = comparison_bbox_df["x_max"] <= image.width
comparison_bbox_df["is_y_in_image"] = comparison_bbox_df["y_max"] <= image.height
comparison_bbox_df["center_x_ratio"] = (
    comparison_bbox_df["bbox_x"] + comparison_bbox_df["bbox_width"] / 2
) / image.width
comparison_bbox_df["center_y_ratio"] = (
    comparison_bbox_df["bbox_y"] + comparison_bbox_df["bbox_height"] / 2
) / image.height
comparison_bbox_df["class_id"] = original_row["class_id"]
comparison_bbox_df["class_name"] = original_row["class_name"]

display(
    comparison_bbox_df[
        [
            "case",
            "class_id",
            "class_name",
            "bbox_x",
            "bbox_y",
            "bbox_width",
            "bbox_height",
            "x_max",
            "y_max",
            "center_x_ratio",
            "center_y_ratio",
            "is_x_in_image",
            "is_y_in_image",
        ]
    ]
)
print(f"Image size: {image.width} x {image.height}")


## 7. 전체 이미지 검수

### 학습 이미지 하나하나 이하 포인트를 맞춰 검수합니다(2차 때 진행).

- 어둡거나 흐림(알약 음각이 잘 인식될까?)
- 반사나 그림자(알약 음각이 잘 인식될까?)
- 알약이 이미지 경계에 잘려 있는지
- 배경 다양성
- 회전정도
- 탐지 대상 오브젝트가 겹쳐있는지
- 어노테이션이 있는데 해당 이미지 없는경우 / 이미지는 있는데 해당 어노테이션이 없는경우
- 어노테이션 (bbox, 클래스) 라벨이 잘못 라벨링 되어있는 경우(bbox 이상치 1건 발견)
- 같은 class인데 촬영 각도나 색감 차이가 큰 경우
- 다른 class인데 외형이 매우 유사한 경우
